# 03. Direct Preference DPO IPO KTO SimPO DNO and SPO

- **Category:** 08_learning_paradigms_rl_and_alignment / 03_llm_alignment_and_reasoning
- **Lineage & Citations:** Rafailov et al. (2023 DPO); Azar et al. (2023 IPO); Ethayarajh et al. (2024 KTO); Meng et al. (2024 SimPO).
- **Purpose:** First-principles mathematical implementation and empirical verification of direct preference optimization algorithms: Direct Preference Optimization (DPO), Identity Preference Optimization (IPO), Simple Preference Optimization (SimPO), and Kahneman-Tversky Optimization (KTO).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [02. RLHF Reward Modeling and PPO vs RLOO](02_rlhf_reward_modeling_and_ppo_vs_rloo.ipynb) $\to$ `🔗 Downstream Syntheses:` [04. GRPO DAPO and Verifiable Reward Reasoning](04_grpo_dapo_and_verifiable_reward_reasoning.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to Use**  
> Never use standard DPO on open-ended creative tasks where response lengths vary widely without length normalization; unnormalized DPO systematically develops severe verbosity bias. In KTO, never treat desirable and undesirable samples symmetrically; human behavioral feedback requires asymmetric loss aversion weighting ($\lambda_U > \lambda_D$).

## Mathematical Foundations & Governing Equations

### 1. Direct Preference Optimization (DPO)
$$
\mathcal{L}_{\text{DPO}}(\theta) = \mathbb{E}_{(x, y_w, y_l)} \left[ \text{softplus}\left( -\beta \left( \left(\ln \pi_\theta(y_w) - \ln \pi_{\text{ref}}(y_w)\right) - \left(\ln \pi_\theta(y_l) - \ln \pi_{\text{ref}}(y_l)\right) \right) \right) \right]
$$

### 2. Identity Preference Optimization (IPO)
$$
\mathcal{L}_{\text{IPO}}(\theta) = \mathbb{E} \left[ \left( \beta \left( \Delta \ln \frac{\pi_\theta}{\pi_{\text{ref}}} \right) - \frac{1}{2\tau} \right)^2 \right]
$$

### 3. Simple Preference Optimization (SimPO)
$$
\bar{p}_\theta(y) = \frac{1}{|y|} \sum_{t=1}^{|y|} \ln \pi_\theta(y_t \mid y_{<t})
$$
$$
\mathcal{L}_{\text{SimPO}}(\theta) = \mathbb{E} \left[ \text{softplus}\left( -\left( \beta (\bar{p}_\theta(y_w) - \bar{p}_\theta(y_l)) - \gamma \right) \right) \right]
$$

### 4. Kahneman-Tversky Optimization (KTO)
$$
\mathcal{L}_{\text{KTO}}(\theta) = \mathbb{E} \left[ w(y) \cdot \sigma\left( \text{sign}(y) \cdot \beta \ln \frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)} \right) \right]
$$

### Asymptotic Time & Memory Complexity
- DPO Loss: $\mathcal{O}(B)$ time and memory.
- IPO Loss: $\mathcal{O}(B)$ time and memory.
- SimPO Loss: $\mathcal{O}(B)$ time and memory (Zero reference model memory!).
- KTO Loss: $\mathcal{O}(B)$ time and memory (Zero pairing requirement!).

In [ ]:
# Setup and seed verification environment
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)

# Locate the golden checkpoint
CKPT_REL = Path("weights/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/03_direct_preference_dpo_ipo_kto_simpo_dno_and_spo.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, w: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return x @ w + b

def dpo_loss(
    pi_logps_w: torch.Tensor,
    pi_logps_l: torch.Tensor,
    ref_logps_w: torch.Tensor,
    ref_logps_l: torch.Tensor,
    beta: float = 0.1,
) -> torch.Tensor:
    pi_diff = pi_logps_w - pi_logps_l
    ref_diff = ref_logps_w - ref_logps_l
    logits = beta * (pi_diff - ref_diff)
    return torch.mean(F.softplus(-logits))

def ipo_loss(
    pi_logps_w: torch.Tensor,
    pi_logps_l: torch.Tensor,
    ref_logps_w: torch.Tensor,
    ref_logps_l: torch.Tensor,
    beta: float = 0.1,
    tau: float = 0.5,
) -> torch.Tensor:
    pi_diff = pi_logps_w - pi_logps_l
    ref_diff = ref_logps_w - ref_logps_l
    logits = beta * (pi_diff - ref_diff)
    target = 1.0 / (2.0 * tau)
    return torch.mean((logits - target) ** 2)

def simpo_loss(
    pi_logps_w: torch.Tensor,
    pi_logps_l: torch.Tensor,
    len_w: torch.Tensor,
    len_l: torch.Tensor,
    beta: float = 2.0,
    gamma_margin: float = 0.5,
) -> torch.Tensor:
    norm_w = pi_logps_w / len_w
    norm_l = pi_logps_l / len_l
    logits = beta * (norm_w - norm_l) - gamma_margin
    return torch.mean(F.softplus(-logits))

def kto_loss(
    pi_logps: torch.Tensor,
    ref_logps: torch.Tensor,
    desirable_flags: torch.Tensor,
    beta: float = 0.1,
    desirable_weight: float = 1.0,
    undesirable_weight: float = 1.33,
) -> torch.Tensor:
    log_ratio = beta * (pi_logps - ref_logps)
    loss_desirable = torch.sigmoid(-log_ratio) * desirable_weight
    loss_undesirable = torch.sigmoid(log_ratio) * undesirable_weight
    per_sample_loss = torch.where(desirable_flags > 0.5, loss_desirable, loss_undesirable)
    return torch.mean(per_sample_loss)

In [ ]:
# Multi-track numerical parity assertions
y = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
err_anchor = float(torch.max(torch.abs(y - weights["test_output"])))
assert err_anchor < 1e-5, f"Linear anchor parity error: {err_anchor:.2e}"

dpo_l = dpo_loss(weights["pi_logps_w"], weights["pi_logps_l"], weights["ref_logps_w"], weights["ref_logps_l"], beta=0.1)
err_dpo = float(torch.abs(dpo_l - weights["dpo_loss_ref"]))
assert err_dpo < 1e-5, f"DPO loss parity error: {err_dpo:.2e}"

ipo_l = ipo_loss(weights["pi_logps_w"], weights["pi_logps_l"], weights["ref_logps_w"], weights["ref_logps_l"], beta=0.1, tau=0.5)
err_ipo = float(torch.abs(ipo_l - weights["ipo_loss_ref"]))
assert err_ipo < 1e-5, f"IPO loss parity error: {err_ipo:.2e}"

simpo_l = simpo_loss(weights["pi_logps_w"], weights["pi_logps_l"], weights["len_w"], weights["len_l"], beta=2.0, gamma_margin=0.5)
err_simpo = float(torch.abs(simpo_l - weights["simpo_loss_ref"]))
assert err_simpo < 1e-5, f"SimPO loss parity error: {err_simpo:.2e}"

kto_l = kto_loss(weights["pi_logps_unpaired"], weights["ref_logps_unpaired"], weights["desirable_flags"], beta=0.1)
err_kto = float(torch.abs(kto_l - weights["kto_loss_ref"]))
assert err_kto < 1e-5, f"KTO loss parity error: {err_kto:.2e}"

print(f"All parity assertions passed successfully (max error {max(err_anchor, err_dpo, err_ipo, err_simpo, err_kto):.2e})")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

b_dpo = benchmark_fn(
    lambda: dpo_loss(weights["pi_logps_w"], weights["pi_logps_l"], weights["ref_logps_w"], weights["ref_logps_l"]),
    iters=100,
    name="Direct Preference Optimization (DPO)",
)
b_ipo = benchmark_fn(
    lambda: ipo_loss(weights["pi_logps_w"], weights["pi_logps_l"], weights["ref_logps_w"], weights["ref_logps_l"]),
    iters=100,
    name="Identity Preference Optimization (IPO)",
)
b_simpo = benchmark_fn(
    lambda: simpo_loss(weights["pi_logps_w"], weights["pi_logps_l"], weights["len_w"], weights["len_l"]),
    iters=100,
    name="Simple Preference Optimization (SimPO)",
)
b_kto = benchmark_fn(
    lambda: kto_loss(weights["pi_logps_unpaired"], weights["ref_logps_unpaired"], weights["desirable_flags"]),
    iters=100,
    name="Kahneman-Tversky Optimization (KTO)",
)

print(format_benchmark_table([b_dpo, b_ipo, b_simpo, b_kto]))

In [ ]:
# Failure Mode Contrast: Verbosity Exploitation in Unnormalized DPO vs Length-Invariance in SimPO
# Suppose winning completion w is concise: length 10, total log_prob = -10.0 (average = -1.0/token)
# Losing completion l is verbose and rambling: length 100, total log_prob = -50.0 (average = -0.5/token)
log_w = torch.tensor([-10.0])
len_w = torch.tensor([10.0])
log_l = torch.tensor([-50.0])
len_l = torch.tensor([100.0])

# In unnormalized DPO, raw difference is: -10 - (-50) = +40.0
raw_diff = (log_w - log_l).item()

# In SimPO with length normalization, normalized rates are:
norm_w = (log_w / len_w).item()  # -1.0
norm_l = (log_l / len_l).item()  # -0.5
norm_diff = norm_w - norm_l      # -0.5 (detects that response l had higher per-token quality!)

print(f"Unnormalized DPO Preference Difference: {raw_diff:.4f} (Massively skewed by length!)")
print(f"SimPO Length-Normalized Rate Diff:       {norm_diff:.4f} (Faithfully reflects per-token density!)")

assert raw_diff > 0.0, f"Raw diff should be positive (got {raw_diff})"
assert norm_diff < 0.0, f"Length normalized diff reflects true per-token density (got {norm_diff})"
print("Pathological failure mode contrast successfully verified")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Implicit Reward Optimization**: DPO demonstrates that reinforcement learning with human feedback can be solved analytically as binary classification without an explicit reward model.
2. **Margin Regularization**: IPO replaces the saturating logistic loss with a quadratic penalty, preventing greedy over-fitting on easy preference pairs.
3. **Reference-Free SimPO**: Length-normalized log-probabilities completely eliminate length bias while removing the reference model from GPU memory.
4. **Unpaired Prospect Optimization**: KTO expands direct preference optimization to un-paired binary user interactions (upvotes/downvotes) via asymmetric loss weighting.

---
### Cross-Domain Generalisation Matrix
- **Open-Source Chatbot Alignment**: DPO and SimPO train lightweight instruction models on single GPU nodes without RL complexity.
- **Code Generation Optimization**: SimPO length-normalization prevents language models from generating bloated boilerplate code.
- **User Feedback Ingestion in Production**: KTO processes streaming website thumbs-up/down signals without offline pair-matching pipelines.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [03_direct_preference_dpo_ipo_kto_simpo_dno_and_spo.md](../../../markdown/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/03_direct_preference_dpo_ipo_kto_simpo_dno_and_spo.md)
- **PyTorch 2.x Module:** [03_direct_preference_dpo_ipo_kto_simpo_dno_and_spo.py](../../../pytorch2/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/03_direct_preference_dpo_ipo_kto_simpo_dno_and_spo.py)
- **Functional JAX Module:** [03_direct_preference_dpo_ipo_kto_simpo_dno_and_spo.py](../../../jax/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/03_direct_preference_dpo_ipo_kto_simpo_dno_and_spo.py)
- **Native Rust Candle Track:** [03_direct_preference_dpo_ipo_kto_simpo_dno_and_spo.rs](../../../candle/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/03_direct_preference_dpo_ipo_kto_simpo_dno_and_spo.rs)
- **Hardware Kernel (CPU / WGSL):** [03_direct_preference_dpo_ipo_kto_simpo_dno_and_spo.rs](../../../kernels/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/03_direct_preference_dpo_ipo_kto_simpo_dno_and_spo.rs)
- **Golden SafeTensors:** [03_direct_preference_dpo_ipo_kto_simpo_dno_and_spo.safetensors](../../../weights/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/03_direct_preference_dpo_ipo_kto_simpo_dno_and_spo.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 02. RLHF Reward Modeling and PPO vs RLOO](02_rlhf_reward_modeling_and_ppo_vs_rloo.ipynb) | [Master Index](../../../README.md#08_learning_paradigms_rl_and_alignment) | [Next: 04. GRPO DAPO and Verifiable Reward Reasoning](04_grpo_dapo_and_verifiable_reward_reasoning.ipynb) &rarr;